### Welcome to Week 6 Day 3!

Let's experiment with a bunch more MCP Servers

In [4]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio, create_static_tool_filter
import os
from IPython.display import Markdown, display
from datetime import datetime
load_dotenv(override=True)

True

### The first type of MCP Server: runs locally, everything local

Here's a really interesting one: a knowledge-graph based memory.

It's a persistent memory store of entities, observations about them, and relationships between them.

https://github.com/modelcontextprotocol/servers/tree/main/src/memory


In [ ]:
""" params = {"command": "npx","args": ["-y", "mcp-memory-libsql"],"env": {"LIBSQL_URL": "file:./memory/ed.db"}}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools """

In [7]:
params = {
      "command": "npx",
      "args": [
        "-y",
        "@modelcontextprotocol/server-memory"
      ],
      "env": {
        "MEMORY_FILE_PATH": "/home/martin/Python/Projects/Github/agents/6_mcp/_solution/memory/memory.jsonl"
  }
}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='create_entities', title='Create Entities', description='Create multiple new entities in the knowledge graph', inputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'The name of the entity'}, 'entityType': {'type': 'string', 'description': 'The type of the entity'}, 'observations': {'type': 'array', 'items': {'type': 'string'}, 'description': 'An array of observation contents associated with the entity'}}, 'required': ['name', 'entityType', 'observations']}}}, 'required': ['entities']}, outputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string', 'description': 'The name of the entity'}, 'entityType': {'type': 'string', 'description': 'The type of the entity'}, 'observations': {'

In [5]:
instructions = "You use your entity tools as a persistent memory to store and recall information about your conversations."
request = "My name's Ed. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP protocol. \
MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
model = "gpt-4.1-mini"

In [8]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Hi Ed! I've noted that you are an LLM engineer teaching a course about AI Agents, which covers the MCP protocol. The MCP protocol connects agents with tools, resources, and prompt templates, making it easy to integrate AI agents with various capabilities. How can I assist you further with this?

In [9]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "My name's Ed. What do you know about me?")
    display(Markdown(result.final_output))

I know that you are an LLM engineer named Ed. You are also teaching a course about AI Agents. Is there anything specific you would like to update or add about yourself?

### Check the trace:

https://platform.openai.com/traces

### The 2nd type of MCP server - runs locally, calls a web service

### Brave Search - apologies - this will need another API key! But it's free again.

https://brave.com/search/api/

Set up your account, and put your key in the .env under `BRAVE_API_KEY`

In [20]:
import json
load_dotenv(override=True)
env = {"BRAVE_API_KEY": os.getenv("BRAVE_SEARCH_API_KEY")}
params = {"command": "npx", "args": ["-y", "@brave/brave-search-mcp-server"], "env": env}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

disabled_tools = create_static_tool_filter(blocked_tool_names=['brave_news_search','brave_image_search','brave_video_search','brave_local_search'])

In [23]:
mcp_tools

[Tool(name='brave_web_search', title='brave_web_search', description='\n    Performs web searches using the Brave Search API and returns comprehensive search results with rich metadata.\n\n    When to use:\n        - General web searches for information, facts, or current topics\n        - Location-based queries (restaurants, businesses, points of interest)\n        - News searches for recent events or breaking stories\n        - Finding videos, discussions, or FAQ content\n        - Research requiring diverse result types (web pages, images, reviews, etc.)\n\n    Returns a JSON list of web results with title, description, and URL.\n    \n    When the "results_filter" parameter is empty, JSON results may also contain FAQ, Discussions, News, and Video results.\n', inputSchema={'$schema': 'http://json-schema.org/draft-07/schema#', 'type': 'object', 'properties': {'query': {'type': 'string', 'maxLength': 400, 'description': 'Search query (max 400 chars, 50 words)'}, 'country': {'default':

In [21]:
instructions = "You are able to search the web for information and briefly summarize the takeaways."
request = f"Please research the latest news on Amazon stock price and briefly summarize its outlook. \
For context, the current date is {datetime.now().strftime('%Y-%m-%d')}"
model = "gpt-4o-mini"

In [22]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30, tool_filter=disabled_tools) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Here's a brief summary of the latest news on Amazon's stock price and its outlook as of April 19, 2026:

1. **Current Price and Performance**: Amazon's stock (AMZN) closed at **$250.56** on April 17, 2026, reflecting an **8.5% increase** year-to-date. It is close to its all-time high of $258.60 reached in November 2025.

2. **Strong Revenue Growth**: For Q4 2025, Amazon reported revenues of **$213.4 billion**, up **14%** year-over-year, with AWS (Amazon Web Services) contributing **$35.6 billion**, a **24% increase**. Advertising revenue also rose by **23%**.

3. **Future Expectations**: Analysts predict a **15.62% upside** potential, forecasting the price could reach around **$289.71** by April 2027. The upcoming Q1 earnings report on April 29, 2026, will be crucial for assessing AWS growth.

4. **Strategic Investments**: Amazon has made significant commitments, including a **$200 billion investment in AI infrastructure** and an **$11.6 billion satellite acquisition**, marking these as pivotal for future growth.

5. **Market Sentiment**: Although the broader market has seen fluctuations, Amazon's stock has defied trends and grown. Analysts encourage long-term investment strategies due to the stock's historical resilience.

In summary, while Amazon faces challenges amid substantial investments, its solid revenue growth and strategic positioning in cloud computing and advertising continue to create a favorable outlook for investors.

### As usual, check out the trace:

https://platform.openai.com/traces

## And now the third type: running remotely

It's actually really hard to find a "remote MCP server" aka "hosted MCP server" aka "managed MCP server".

It's not a common model for using or sharing MCP servers, and there isn't a standard way to discover remote MCP servers.

Anthropic lists some remote MCP servers, but these are for paid applications with business users:

https://docs.anthropic.com/en/docs/agents-and-tools/remote-mcp-servers

CloudFlare has tooling for you to create and deploy your own remote MCP servers, but this does not seem to be a common practice:

https://developers.cloudflare.com/agents/guides/remote-mcp-server/


# And back to the 2nd type: the Polygon.io MCP Server

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">PLEASE READ!!-</h2>
            <span style="color:#ff7800;">This service for financial market data has both a FREE plan and a PAID plan, and we can use either depending on your appetite.
            </span>
        </td>
    </tr>
</table>

## NEW SECTION: Introducing polygon.io

Polygon.io is a hugely popular financial data provider. It has a free plan and a paid plan. And it also has an MCP Server!

First, read up on polygon.io on their excellent website, including looking at their pricing:

https://polygon.io

### Polygon.io Part 1: Polygon.io free service (the paid will be totally optional, of course!)

1. Please sign up for polygon.io (top right)  
2. Once signed in, please select "Keys" in the left hand navigation
3. Press the blue "New Key" button
4. Copy the key name
5. Edit your .env file and add the row:

`POLYGON_API_KEY=xxxx`

In [8]:
load_dotenv(override=True)
massive_api_key = os.getenv("MASSIVE_API_KEY")
if not massive_api_key:
    print("POLYGON_API_KEY is not set")

In [26]:
from massive import RESTClient
client = RESTClient(massive_api_key)
client.get_previous_close_agg("AAPL")[0]

PreviousCloseAgg(ticker='AAPL', close=270.23, high=272.3, low=266.72, open=266.96, timestamp=1776456000000, volume=61436228.0, vwap=269.8735)

### Wrapped into a python module that caches end of day prices

I've made a python module `market.py` that uses this API to look up share prices.

But the free API is quite heavily rate limited - so I've been a bit sneaky; when you ask for a share price, this function retrieves the entire end-of-day equity market, and caches it in our database.


In [1]:
from market import get_share_price
get_share_price("AAPL")

270.23

In [2]:
# no rate limiting concerns!

for i in range(1000):
    get_share_price("AAPL")
get_share_price("AAPL")

270.23

### And I've made this into an MCP Server

Just as we did with accounts.py; see `market_server.py`

In [5]:
params = {"command": "uv", "args": ["run", "market_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools

[Tool(name='lookup_share_price', title=None, description='This tool provides the current price of the given stock symbol.\n\n    Args:\n        symbol: the symbol of the stock\n    ', inputSchema={'properties': {'symbol': {'title': 'Symbol', 'type': 'string'}}, 'required': ['symbol'], 'title': 'lookup_share_priceArguments', 'type': 'object'}, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'number'}}, 'required': ['result'], 'title': 'lookup_share_priceOutput', 'type': 'object'}, icons=None, annotations=None, meta=None, execution=None)]

### Let's try it out!

Hopefully gpt-4o-mini is smart enough to know that the symbol for Apple is AAPL

In [6]:
instructions = "You answer questions about the stock market."
request = "What's the share price of Apple?"
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

The current share price of Apple (AAPL) is $270.23.

## Polygon.io Part 2: Paid Plan - Totally Optional!

If you are interested, you can subscribe to the monthly plan to get more up to date market data, and unlimited API calls.

If you do wish to do this, then it also makes sense to use the full MCP server that Polygon.io has released, to take advantage of all their functionality.



In [9]:

params = {"command": "uvx",
          "args": ["--from", "git+https://github.com/massive-com/mcp_massive@v0.9.1", "mcp_massive"],
          "env": {"MASSIVE_API_KEY": massive_api_key}
          }
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools


[Tool(name='search_endpoints', title=None, description='Search for market data API endpoints and built-in finance functions by natural language query. Use this FIRST to find the right endpoint before calling call_api. Covers stocks, options, forex, crypto, futures, indices, ETFs, and economic data. Pass market to pin results to a specific asset class when you already know it; omit it and the server will infer from the query. Use detail="more" to see query parameter docs needed for building call_api requests.', inputSchema={'additionalProperties': False, 'properties': {'query': {'description': 'Natural language search query for API endpoints', 'minLength': 1, 'title': 'Query', 'type': 'string'}, 'scope': {'anyOf': [{'enum': ['all', 'endpoints', 'functions'], 'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Search scope: "endpoints" for API only, "functions" for local functions only, or "all"/omit for both', 'title': 'Scope'}, 'max_results': {'anyOf': [{'maximum': 2

### Wow that's a lot of tools!

Let's try them out - hopefully the sheer number of tools doesn't overwhelm gpt-4o-mini!

With the $29 monthly plan, we don't have access to some of the APIs, so I've needed to specify which APIs can be called.

If you've splashed out on a bigger plan, feel free to remove my extra constraint..

In [10]:
instructions = "You answer questions about the stock market."
request = "What's the share price of Apple? Use your tools to get the latest price."
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

I am currently unable to access the real-time data for Apple's share price due to authorization restrictions. However, you can check the latest price on major financial websites like Yahoo Finance, Google Finance, or your brokerage platform. If you want, I can provide you with the historical data or other stock information that I have access to. Would you like me to do that?

## Setting up your .env file

If you do decide to have a paid plan, please add this to your .env file to indicate:

`POLYGON_PLAN=paid`

And if you decide to go all the way for the realtime API, then please do:

`POLYGON_PLAN=realtime`

In [ ]:
load_dotenv(override=True)

polygon_plan = os.getenv("POLYGON_PLAN")
is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

if is_paid_polygon:
    print("You've chosen to subscribe to the paid Polygon plan, so the code will look at prices on a 15 min delay")
elif is_realtime_polygon:
    print("Wowzer - you've chosen to subscribe to the realtime Polygon plan, so the code will look at realtime prices")
else:
    print("According to your .env file, you've chosen to subscribe to the free Polygon plan, so the code will look at EOD prices")

## And that's it for today!

I've removed the part of this lab that uses the "Financial Datasets" mcp server, because it's inferior - more expensive with fewer APIs.

And this way we get to use the same provider for Free and Paid APIs.

But if you want to see the code, just look in the git history for a prior version.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Explore MCP server marketplaces and integrate your own, using all 3 approaches.
            </span>
        </td>
    </tr>
</table>